# Milestone 4: Training Runs on Kaggle

One notebook for every training run. List the runs to do in `RUN_IDS` (each is a file in `configs/`); they run one after another in this session, so a whole queue runs overnight. Then use **Save Version → Save & Run All**.

**Needs:** GPU (T4, or T4 ×2 for the final runs), Internet on, and the dataset `dsai-group3-m4-data` attached. Keep a queue under about 11 hours: a Kaggle session stops at 12.


In [ ]:
RUN_IDS = ["default"]     # e.g. ["T-02", "T-03", "T-04"]; each is configs/<RUN_ID>.yaml
BRANCH = "milestone-4"
RESUME = False            # True to continue the first run in the list from its last checkpoint

In [ ]:
!git clone -q --depth 1 -b {BRANCH} https://github.com/thehimanshhu/Group-3-DS-and-AI-Lab-Project repo
!pip install -q -r repo/requirements.txt
!git -C repo rev-parse --short HEAD

In [ ]:
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import yaml

# train.py silently falls back to CPU, so fail here instead of losing hours
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2 and restart the session'
print(f"GPU: {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}")

manifests = list(Path('/kaggle/input').rglob('manifest_m4.csv'))
assert manifests, 'Attach the dataset dsai-group3-m4-data'

DATA_DIR = manifests[0].parent.parent
missing = [r for r in RUN_IDS if not Path(f'repo/configs/{r}.yaml').exists()]
assert not missing, f'No config for {missing}: merge it into {BRANCH} first'

In [ ]:
def channel_enabled(run_id):
    cfg = yaml.safe_load(Path(f'repo/configs/{run_id}.yaml').read_text())
    return cfg.get('augment', {}).get('channel', {}).get('enabled', False)

# V-09/V-10 need ffmpeg for the phone channel; a missing binary fails those runs mid-queue
needs_ffmpeg = [r for r in RUN_IDS if channel_enabled(r)]
assert not needs_ffmpeg or shutil.which('ffmpeg'), f'ffmpeg missing; needed by {needs_ffmpeg}'
print('ffmpeg:', shutil.which('ffmpeg') or 'not needed for this queue')

In [ ]:
# Quick smoke test: every enabled augmentation on one real clip, forced on (p=1)
sys.path.insert(0, str(Path('repo').resolve()))
from src.augment import apply

manifest = pd.read_csv(DATA_DIR / 'manifests' / 'manifest_m4.csv', keep_default_na=False)
row = manifest[manifest.split == 'train'].sort_values('duration_s', ascending=False).iloc[0]
wave, sr = sf.read(DATA_DIR / row.path, frames=4 * 16000, dtype='float32')

for run_id in RUN_IDS:
    augment_cfg = yaml.safe_load(Path(f'repo/configs/{run_id}.yaml').read_text()).get('augment', {})
    enabled = [name for name, effect in augment_cfg.items() if isinstance(effect, dict) and effect.get('enabled')]
    
    if enabled:
        forced = {name: {**effect, 'p': 1.0} if isinstance(effect, dict) else effect
                  for name, effect in augment_cfg.items()}
        out = apply(wave, sr, forced, np.random.default_rng(0))
        
        assert out.size and np.isfinite(out).all(), f'{run_id}: augmentation produced invalid audio'
        print(f"{run_id}: augmentation OK ({', '.join(enabled)})")

print('preflight OK |', DATA_DIR, '| queue:', RUN_IDS)

In [ ]:
import subprocess, time

for i, run_id in enumerate(RUN_IDS):
    out_dir = Path('/kaggle/working') / run_id
    cmd = ['python', '-m', 'src.train', '--config', f'configs/{run_id}.yaml',
           '--data', str(DATA_DIR), '--out', str(out_dir)]
    if RESUME and i == 0:
        cmd.append('--resume')
    start = time.time()
    result = subprocess.run(cmd, cwd='repo')
    print(f'{run_id}: exit code {result.returncode}, {(time.time() - start) / 60:.0f} min')
    # A failed run doesn't stop the queue; its folder shows what happened.

In [ ]:
import json
import pandas as pd

rows = [json.load(open(p)) for run_id in RUN_IDS if (p := Path('/kaggle/working') / run_id / 'run.json').exists()]
print(pd.DataFrame(rows), flush=True)

**After the queue:**

1. For each run, download the small files from this notebook's output (`run.json`, `config.yaml`, `train_log.csv`, `epoch_log.csv`, `valid_predictions.csv`) into `results/<RUN_ID>/` on your branch and open a PR. `predictions.csv` and `results.json` come from `src/evaluate.py`, not from this notebook, so they won't be here. Tip: `cd /kaggle/working && zip -r small_results.zip */run.json */config.yaml */train_log.csv */epoch_log.csv */valid_predictions.csv` bundles every run's small files.
2. Fill in `run.json`'s `kaggle_notebook` with this notebook's URL (`train.py` leaves it empty), then run `python -m src.collect_runs` to rebuild `results/runs.csv`.
3. `best.pt` stays here, in the notebook output. Share this notebook with the team so it can be attached as an input.
4. Tick the runs on the "M4 run tracker" issue and post their `valid` EER on WhatsApp.
